# **Training Query Intent Classifier**
Notebook documenting the training of an agent to classify code execution intent from a user query.

---

In [ ]:
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'

### **Load Both Datasets**

In [ ]:
# SKILLRET_ds = load_dataset('ThakiCloud/SKILLRET', 'queries', split='train').to_pandas()
SKILLRET_ds = pd.read_json('https://huggingface.co/datasets/ThakiCloud/SKILLRET/resolve/main/data/queries/train.jsonl', lines=True, )

In [ ]:
MS_MARCO_ds = load_dataset('microsoft/ms_marco', 'v1.1', split='train', streaming=True)
MS_MARCO_ds = MS_MARCO_ds.shuffle(seed=42).take(64000)
MS_MARCO_ds = pd.DataFrame(list(MS_MARCO_ds))

### **Synthesize New Dataset**
create a dataset by merging the two dataset's queries and a 'code_intent' column which is 1 for SKILLRET_ds and 0 for MS_MARCO_ds

In [ ]:
df = pd.concat([SKILLRET_ds[['query']].assign(code_intent=1), MS_MARCO_ds[['query']].assign(code_intent=0)], ignore_index=True)

# explicit free memory
del SKILLRET_ds
# del MS_MARCO_ds

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.describe()

### **Dataset Sanity Check**

In [ ]:
print('number of null values')
df.isnull().sum()

### **Preprocessing**

Setting up tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
encodings = tokenizer(
    df['query'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=64,
)

Assign the tokenized vectors to dataframe

In [ ]:
df['query'] = encodings['input_ids']
df['code_intent'] = encodings['attention_mask']

In [ ]:
df.head()

### **Split Dataset**

In [ ]:
df = train_test_split(df, test_size=0.2, random_state=42)

### **Configure DistilBERT Model**

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

Define training arguments

In [ ]:
args = TrainingArguments(
    output_dir="../models/intent-classifier",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

### **Train Model**

Define a function to compute evaluation metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", pos_label=1
    )
    acc = accuracy_score(labels, predictions)
    return {
        "accuracy":  round(acc, 4),
        "precision": round(precision, 4),
        "recall":    round(recall, 4),
        "f1":        round(f1, 4),
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=df['train'],
    eval_dataset=df['test'],
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()